# 🏨 Playground · Exploración de datos (EDA)

⚠️ Este notebook es **autónomo**: **no** importa nada de `src/` ni de `ml_hotel_cancellations`. Todo el código vive aquí, igual que en los notebooks de `recursos/`. Aquí *aprendimos practicando*; de esta exploración salieron luego las decisiones del paquete `src/`.

> Cargamos el dataset **crudo** y exploramos **antes de tocar nada** (incluidas `company`, `agent` y `arrival_date_year`). La limpieza y las decisiones de modelado se recogen al final, en *Conclusiones y decisiones*.

## 0. Configuración del notebook

Importamos todas las librerías al principio (estilo `recursos/`) para tenerlas en un único sitio. Trabajaremos sobre todo con **Plotly** para los gráficos interactivos, incrustados en línea (`pio.renderers.default = "notebook_connected"`).

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import plotly.io as pio

pio.renderers.default = "notebook_connected"
import warnings; warnings.filterwarnings('ignore')

# Paleta de color consistente por hotel en todo el notebook
COLOR_HOTEL = {"City Hotel": "#d95f02", "Resort Hotel": "#2c7fb8"}
# Orden natural de los meses (el dataset los trae como texto en inglés)
ORDEN_MESES = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

### Función auxiliar: matriz de correlación

Reutilizamos el helper de clase (`recursos/clase_1`). Dibuja un mapa de calor interactivo de las correlaciones entre variables **numéricas** y, además, imprime el *ranking* de correlaciones con la variable dependiente.

In [2]:
def plot_matriz_correlacion(df: pd.DataFrame, variable_dependiente: str) -> None:
    """Dibuja una matriz de correlación interactiva con Plotly.

    Crea un mapa de calor con las correlaciones entre todas las variables
    numéricas del DataFrame (anotando los valores) e imprime el ranking de
    correlaciones con la variable dependiente indicada.

    Args:
        df (pd.DataFrame): DataFrame con las variables numéricas a analizar.
        variable_dependiente (str): Columna objetivo cuyas correlaciones se
            mostrarán ordenadas.

    Returns:
        None: muestra el gráfico interactivo y la tabla de correlaciones.

    Example:
        >>> plot_matriz_correlacion(df_numericas, "is_canceled")
    """
    # Matriz de correlación entre variables numéricas
    correlation_matrix = df.corr()

    # Mapa de calor interactivo
    fig = px.imshow(
        correlation_matrix,
        text_auto='.2f',                    # Mostrar valores con 2 decimales
        color_continuous_scale='RdBu_r',    # Rojo-blanco-azul invertido
        zmin=-1, zmax=1,                    # Rango fijo de correlación
        aspect="auto",
        title='Matriz de correlación entre variables',
    )
    fig.update_layout(
        width=800,
        height=700,
        coloraxis_colorbar=dict(
            title="Coeficiente<br>de correlación",
            thicknessmode="pixels", thickness=20,
            lenmode="pixels", len=500,
            yanchor="top", y=1,
            ticks="outside",
        ),
        font=dict(size=12),
    )

    # Correlaciones con la variable dependiente, ordenadas
    correlaciones_con_dependiente = correlation_matrix[variable_dependiente].sort_values(ascending=False)

    fig.show()

    print(f"\nCorrelaciones con '{variable_dependiente}':")
    print(correlaciones_con_dependiente)

### Función auxiliar: tasa de cancelación por categoría

La forma más informativa de mirar una variable categórica frente a la clase es la **tasa de cancelación**: la media de `is_canceled` dentro de cada categoría. Este helper devuelve una tabla ordenada (con conteos) y, opcionalmente, desglosada **por hotel**.

In [3]:
def tasa_por_categoria(df: pd.DataFrame, col: str, by_hotel: bool = False) -> pd.DataFrame:
    """Calcula la tasa de cancelación por categoría de una columna.

    Args:
        df (pd.DataFrame): DataFrame con la columna ``is_canceled``.
        col (str): Columna categórica a agrupar.
        by_hotel (bool): Si es True, desglosa además por ``hotel``.

    Returns:
        pd.DataFrame: tabla ordenada (descendente por tasa) con la tasa de
        cancelación y el número de reservas de cada categoría.
    """
    claves = [col] if not by_hotel else ["hotel", col]
    tabla = (
        df.groupby(claves)["is_canceled"]
        .agg(tasa_cancelacion="mean", reservas="size")
        .reset_index()
        .sort_values("tasa_cancelacion", ascending=False)
    )
    return tabla

## 1. Carga de datos (crudos)

Leemos el dataset de reservas hoteleras directamente del directorio de datos crudos. Tratamos las cadenas vacías y los marcadores tipo `NULL`/`NA` como valores ausentes (`NaN`). **No** quitamos columnas todavía: queremos explorar todo el dataset tal cual antes de decidir nada.

In [4]:
PATH_DIRECTORIO_DATOS = "../../data/raw"
PATH_DATASET_HOTEL = f"{PATH_DIRECTORIO_DATOS}/dataset_practica_final.csv"

In [5]:
df = pd.read_csv(PATH_DATASET_HOTEL, na_values=["NULL", "NA", "NaN", ""])
df.shape

(119390, 32)

In [6]:
# Primeras filas para hacernos una idea del contenido
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [7]:
# Tipos y conteo de no-nulos por columna
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

### Columnas derivadas

Creamos tres columnas que nos acompañarán durante todo el EDA:

- `has_company`: ¿la reserva tiene una **empresa** asociada? (`company` no nula).
- `has_agent`: ¿la reserva tiene una **agencia** asociada? (`agent` no nulo).
- `noches`: total de noches = noches de semana + noches de fin de semana.

Estas binarias son la idea central de la sección de *ausencia informativa*: convierten un nulo en señal.

In [8]:
df["has_company"] = df["company"].notna()
df["has_agent"] = df["agent"].notna()
df["noches"] = df["stays_in_week_nights"] + df["stays_in_weekend_nights"]

df[["company", "has_company", "agent", "has_agent", "noches"]].head()

,company,has_company,agent,has_agent,noches
0,NaN,False,NaN,False,0
1,NaN,False,NaN,False,0
2,NaN,False,NaN,False,1
3,NaN,False,304.0,True,1
4,NaN,False,240.0,True,2


**Contexto.** Son **119.390 reservas** y **32 columnas** (más las 3 derivadas), repartidas entre **dos hoteles**: un *City Hotel* (urbano) y un *Resort Hotel* (vacacional). Como veremos, se comportan casi como dos negocios distintos.

## 2. La variable objetivo (`is_canceled`)

Es un problema de **clasificación binaria**: `0` = la reserva se mantuvo, `1` = se canceló. Veamos cómo se reparten las clases en global y **por hotel**.

In [9]:
# Frecuencias absolutas y relativas de la clase
print(df["is_canceled"].value_counts())
print()
print(df["is_canceled"].value_counts(normalize=True).round(4))

is_canceled
0    75166
1    44224
Name: count, dtype: int64

is_canceled
0    0.6296
1    0.3704
Name: proportion, dtype: float64


In [10]:
# Reparto global de la clase con etiquetas legibles
serie_objetivo = (
    df["is_canceled"]
    .map({0: "No cancelada", 1: "Cancelada"})
    .value_counts()
    .reset_index()
)
serie_objetivo.columns = ["clase", "reservas"]

fig = px.pie(
    serie_objetivo,
    names="clase",
    values="reservas",
    title="Reparto global de la variable objetivo (is_canceled)",
    color="clase",
    color_discrete_map={"No cancelada": "#2c7fb8", "Cancelada": "#d95f02"},
    width=600, height=450,
)
fig.update_traces(textinfo="percent+label")
fig.show()

In [11]:
# Tasa de cancelación POR HOTEL
tasa_hotel = (
    df.groupby("hotel")["is_canceled"]
    .agg(tasa_cancelacion="mean", reservas="size")
    .reset_index()
)

fig = px.bar(
    tasa_hotel,
    x="hotel",
    y="tasa_cancelacion",
    color="hotel",
    color_discrete_map=COLOR_HOTEL,
    text="tasa_cancelacion",
    title="Tasa de cancelación por hotel",
    labels={"hotel": "Hotel", "tasa_cancelacion": "Tasa de cancelación"},
    width=600, height=420,
)
fig.update_traces(texttemplate="%{text:.1%}")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()
tasa_hotel

,hotel,tasa_cancelacion,reservas
0,City Hotel,0.417270,79330
1,Resort Hotel,0.277634,40060


**Observación.** En global, alrededor del **37 %** de las reservas se cancelan frente a un **63 %** que se mantienen: un **desbalanceo moderado**. Pero el agregado esconde dos realidades muy distintas: el *City Hotel* cancela **≈ 41,7 %** y el *Resort Hotel* **≈ 27,8 %**. El desbalanceo no es extremo, pero sí suficiente para que el *accuracy* engañe → usaremos partición **estratificada** y mediremos con **ROC-AUC**.

## 3. Dos hoteles, casi dos negocios

Antes de seguir, comparemos los dos hoteles en sus rasgos clave. La tabla resume reservas, tasa de cancelación, antelación (`lead_time`), precio medio diario (`adr`), noches, y el porcentaje de reservas con agencia/empresa.

In [12]:
# Resumen comparativo por hotel
resumen_hotel = df.groupby("hotel").agg(
    reservas=("is_canceled", "size"),
    tasa_cancel=("is_canceled", "mean"),
    lead_time_med=("lead_time", "median"),
    adr_med=("adr", "median"),
    noches_med=("noches", "median"),
    pct_con_agent=("has_agent", "mean"),
    pct_con_company=("has_company", "mean"),
).round(3)
resumen_hotel

,reservas,tasa_cancel,lead_time_med,adr_med,noches_med,pct_con_agent,pct_con_company
hotel,,,,,,,
City Hotel,79330,0.417,74.0,99.9,3.0,0.898,0.047
Resort Hotel,40060,0.278,57.0,75.0,3.0,0.795,0.078


**Lectura.** Dos perfiles muy distintos:

- **City Hotel** (urbano): mucho más volumen, **más antelación** media, casi siempre reservado **con agencia** y **cancela más** (≈ 41,7 %).
- **Resort Hotel** (vacacional): estancias algo **más largas**, reservas **más directas** (mucha menos presencia de agencia) y **cancela menos** (≈ 27,8 %).

Conclusión metodológica: a partir de aquí miraremos **casi todo POR HOTEL**, porque agregar los dos puede ocultar o invertir patrones (paradoja de Simpson).

## 4. Valores ausentes

Veamos qué columnas tienen huecos y cuánto, para decidir cómo tratarlas en el preprocesado.

In [13]:
# Porcentaje de valores ausentes por columna (orden descendente)
porcentaje_missing = (df.isna().mean() * 100).round(3)
tabla_missing = (
    porcentaje_missing[porcentaje_missing > 0]
    .sort_values(ascending=False)
    .reset_index()
)
tabla_missing.columns = ["columna", "pct_ausentes"]
tabla_missing

,columna,pct_ausentes
0,company,94.307
1,agent,13.686
2,country,0.409
3,children,0.003


In [14]:
# Gráfico de barras con las columnas que tienen huecos
if tabla_missing.empty:
    print("No hay valores ausentes en el dataset.")
else:
    fig = px.bar(
        tabla_missing.sort_values("pct_ausentes"),
        x="pct_ausentes",
        y="columna",
        orientation="h",
        text="pct_ausentes",
        title="Porcentaje de valores ausentes por columna",
        labels={"pct_ausentes": "% ausentes", "columna": "Columna"},
        color="pct_ausentes",
        color_continuous_scale="OrRd",
        width=700, height=400,
    )
    fig.update_traces(texttemplate="%{text:.2f}%")
    fig.show()

**Comentario.** Cuatro columnas con huecos: `company` (**~94 %** vacía), `agent` (**~14 %**), `country` (**< 1 %**) y `children` (apenas **4 filas**). El caso de `company` y `agent` es especial: su ausencia **no es ruido**, es información. Lo vemos en la sección siguiente.

## 5. Ausencia informativa: `company` y `agent`

Aquí está uno de los hallazgos centrales del EDA. Un nulo **no siempre es un dato perdido**: en `company` y `agent` el hueco significa *"reserva sin empresa / sin agencia"*. Veamos si esa ausencia se relaciona con la cancelación.

In [15]:
# Tasa de cancelación según haya o no company (global)
gap_company = (
    df.groupby("has_company")["is_canceled"]
    .agg(tasa_cancelacion="mean", reservas="size")
    .reset_index()
)
print("COMPANY: %.1f%% de nulos, %d empresas distintas" % (
    df["company"].isna().mean() * 100, df["company"].nunique()))
gap_company

COMPANY: 94.3% de nulos, 352 empresas distintas


,has_company,tasa_cancelacion,reservas
0,False,0.382200,112593
1,True,0.175224,6797


In [16]:
# company x hotel: tasa de cancelación (la señal, desglosada por hotel)
comp_hotel = df.groupby(["hotel", "has_company"])["is_canceled"].mean().reset_index()
comp_hotel["has_company"] = comp_hotel["has_company"].map({True: "Con empresa", False: "Sin empresa"})

fig = px.bar(
    comp_hotel,
    x="hotel",
    y="is_canceled",
    color="has_company",
    barmode="group",
    text="is_canceled",
    title="Tasa de cancelación según tenga empresa (company), por hotel",
    labels={"is_canceled": "Tasa de cancelación", "hotel": "Hotel", "has_company": ""},
    color_discrete_map={"Con empresa": "#1b9e77", "Sin empresa": "#d95f02"},
    width=700, height=440,
)
fig.update_traces(texttemplate="%{text:.1%}")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

**Conclusión (company).** El nulo de `company` **no es aleatorio**: marca reservas *sin empresa / particulares*. Y **tener empresa protege** contra la cancelación: con empresa se cancela mucho menos (≈ 14 %) que sin ella (≈ 38 % global), y el patrón se mantiene **dentro de cada hotel**. Hay información valiosa escondida en ese hueco.

In [17]:
# Tasa de cancelación según haya o no agent (global)
gap_agent = (
    df.groupby("has_agent")["is_canceled"]
    .agg(tasa_cancelacion="mean", reservas="size")
    .reset_index()
)
print("AGENT: %.1f%% de nulos, %d agencias distintas" % (
    df["agent"].isna().mean() * 100, df["agent"].nunique()))
print()
print("%% de reservas con agente por hotel:")
print(df.groupby("hotel")["has_agent"].mean().round(3).to_string())
gap_agent

AGENT: 13.7% de nulos, 333 agencias distintas

%% de reservas con agente por hotel:
hotel
City Hotel      0.898
Resort Hotel    0.795


,has_agent,tasa_cancelacion,reservas
0,False,0.246634,16340
1,True,0.390044,103050


In [18]:
# agent x hotel: tasa de cancelación (ojo: agent está confundido con el hotel)
ag_hotel = df.groupby(["hotel", "has_agent"])["is_canceled"].mean().reset_index()
ag_hotel["has_agent"] = ag_hotel["has_agent"].map({True: "Con agencia", False: "Sin agencia"})

fig = px.bar(
    ag_hotel,
    x="hotel",
    y="is_canceled",
    color="has_agent",
    barmode="group",
    text="is_canceled",
    title="Tasa de cancelación según tenga agencia (agent), por hotel",
    labels={"is_canceled": "Tasa de cancelación", "hotel": "Hotel", "has_agent": ""},
    color_discrete_map={"Con agencia": "#d95f02", "Sin agencia": "#1b9e77"},
    width=700, height=440,
)
fig.update_traces(texttemplate="%{text:.1%}")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

**Conclusión (agent).** Aquí la señal va en **sentido opuesto** al de company: tener agencia se asocia a **más** cancelación. Pero ojo, hay una trampa: `agent` está **confundido con el hotel** (el *City* tiene agencia en ≈ 91 % de las reservas y el *Resort* solo en ≈ 7,5 %). Por eso lo miramos **por hotel**: incluso así, *dentro de cada hotel* con agencia se cancela más. La señal es real, pero hay que tratarla con cuidado.

**Propuesta (para el pipeline `src/`).** En vez de tirar `company` (94 % nula, 352 IDs distintos → demasiada cardinalidad para *one-hot*), proponemos crear dos **features binarias** baratas que capturan la señal:

- `has_company` → *protege* contra la cancelación.
- `has_agent` → *riesgo* (con la advertencia de que está confundida con el hotel).

Esto es solo una **idea** que sale del EDA; aquí no modificamos el pipeline.

## 6. Cobertura temporal: año × semana

¿Cubren los tres años (2015–2017) el mismo periodo? Cruzamos `arrival_date_week_number` (semana del año) con `arrival_date_year` y contamos reservas. Si los años no se solapan, `arrival_date_year` no servirá para generalizar.

In [19]:
# Pivot: número de reservas por (semana del año) x (año)
pivot_anio_semana = df.pivot_table(
    index="arrival_date_week_number",
    columns="arrival_date_year",
    values="is_canceled",
    aggfunc="count",
).sort_index()

fig = px.imshow(
    pivot_anio_semana,
    color_continuous_scale="OrRd",
    aspect="auto",
    title="Cobertura temporal: nº de reservas por semana del año y año",
    labels={"x": "Año", "y": "Semana del año", "color": "Reservas"},
    width=600, height=700,
)
fig.show()

In [20]:
# Rango de semanas con datos por año (resumen del solapamiento)
rango_semanas = (
    df.groupby("arrival_date_year")["arrival_date_week_number"]
    .agg(semana_min="min", semana_max="max", reservas="size")
)
rango_semanas

,semana_min,semana_max,reservas
arrival_date_year,,,
2015,27,53,21996
2016,1,53,56707
2017,1,35,40687


**Conclusión.** Los años son **parciales y desplazados**:

- **2015**: semanas ~27–53 → solo **2º semestre**.
- **2016**: semanas ~1–53 → año **completo**.
- **2017**: semanas ~1–35 → solo **1er semestre**.

Es decir, `arrival_date_year` **no generaliza** (un año no visto no tiene valor interpretable) y además está **confundido con la estación** (2015 = verano/otoño, 2017 = invierno/primavera). Lo descartaremos como predictor; la estacionalidad la capturan `arrival_date_month` y `arrival_date_week_number`.

## 7. Estacionalidad

Dos miradas estacionales, ambas **por hotel**: (a) el **volumen** de reservas a lo largo del año y (b) la **tasa de cancelación** mes a mes.

In [21]:
# (a) Demanda estacional: reservas por semana del año, una línea por hotel
demanda = (
    df.groupby(["hotel", "arrival_date_week_number"])
    .size()
    .reset_index(name="reservas")
)

fig = px.line(
    demanda,
    x="arrival_date_week_number",
    y="reservas",
    color="hotel",
    color_discrete_map=COLOR_HOTEL,
    title="Demanda estacional: reservas por semana del año (por hotel)",
    labels={"arrival_date_week_number": "Semana del año", "reservas": "Reservas", "hotel": "Hotel"},
    width=800, height=450,
)
fig.show()

In [22]:
# (b) Tasa de cancelación por mes, por hotel (meses en orden natural)
tasa_mes = (
    df.groupby(["hotel", "arrival_date_month"])["is_canceled"]
    .mean()
    .reset_index()
)
# Ordenamos el mes como categoría ordenada
tasa_mes["arrival_date_month"] = pd.Categorical(
    tasa_mes["arrival_date_month"], categories=ORDEN_MESES, ordered=True
)
tasa_mes = tasa_mes.sort_values("arrival_date_month")

fig = px.line(
    tasa_mes,
    x="arrival_date_month",
    y="is_canceled",
    color="hotel",
    color_discrete_map=COLOR_HOTEL,
    markers=True,
    title="Tasa de cancelación por mes (por hotel)",
    labels={"arrival_date_month": "Mes", "is_canceled": "Tasa de cancelación", "hotel": "Hotel"},
    width=850, height=450,
)
fig.update_yaxes(range=[0, 0.6], tickformat=".0%")
fig.show()

**Hallazgo central.** Las dos curvas cuentan historias distintas:

- El **Resort** es claramente **estacional**: cancela poco en invierno (≈ 15 % en enero) y mucho más en temporada alta (verano y diciembre, ~33 %). Su cancelación sigue a la demanda vacacional.
- La **City** es **plana y alta** todo el año (~38–43 %): es un hotel de negocio/urbano cuya cancelación apenas depende del mes.

Otro argumento más para mirar (y quizá modelar) los dos hoteles por separado.

## 8. Anomalías y saneamiento de datos

Buscamos registros **imposibles o sospechosos**. Empezamos por las fechas y seguimos con valores numéricos fuera de rango:

- **Año de llegada**: el dataset solo cubre 2015–2017; comprobamos que no haya años posteriores.
- **`adr`** (tarifa media diaria): no debería ser negativa ni absurdamente alta.
- **Huéspedes y noches**: reservas con 0 personas o 0 noches no tienen sentido.
- **Ocupación extrema**: un número desorbitado de niños o bebés.

### 8.1 Fechas

In [23]:
anom_anio = df[df["arrival_date_year"] > 2017]
print("Años de llegada presentes:", sorted(df["arrival_date_year"].unique()))
print(f"Filas con año imposible (> 2017): {len(anom_anio)}  ->  la columna de año está limpia")

Años de llegada presentes: [2015, 2016, 2017]
Filas con año imposible (> 2017): 0  ->  la columna de año está limpia


### 8.2 `adr` (tarifa media diaria) negativa o desorbitada

In [24]:
print("adr  ->  min = %.2f , max = %.2f" % (df["adr"].min(), df["adr"].max()))
print("  adr negativa:", int((df["adr"] < 0).sum()), "| adr > 1000:", int((df["adr"] > 1000).sum()))
df.loc[(df["adr"] < 0) | (df["adr"] > 1000),
       ["hotel", "adr", "is_canceled", "market_segment", "reserved_room_type"]]

adr  ->  min = -6.38 , max = 5400.00
  adr negativa: 1 | adr > 1000: 1


,hotel,adr,is_canceled,market_segment,reserved_room_type
14969,Resort Hotel,-6.38,0,Groups,A
48515,City Hotel,5400.00,1,Offline TA/TO,A


### 8.3 Reservas sin huéspedes, sin noches u ocupación extrema

In [25]:
sin_huespedes = int((df[["adults", "children", "babies"]].fillna(0).sum(axis=1) == 0).sum())
sin_noches = int(((df["stays_in_week_nights"] + df["stays_in_weekend_nights"]) == 0).sum())
ocup_extrema = (df["children"] > 8) | (df["babies"] > 8)
print(f"Reservas con 0 huéspedes: {sin_huespedes}")
print(f"Reservas con 0 noches:    {sin_noches}")
print(f"Ocupación extrema (>8 niños o bebés): {int(ocup_extrema.sum())}")
df.loc[ocup_extrema, ["hotel", "adults", "children", "babies", "adr", "is_canceled"]]

Reservas con 0 huéspedes: 180
Reservas con 0 noches:    715
Ocupación extrema (>8 niños o bebés): 3


,hotel,adults,children,babies,adr,is_canceled
328,Resort Hotel,2,10.0,0,133.16,1
46619,City Hotel,2,0.0,10,84.45,0
78656,City Hotel,1,0.0,9,95.00,0


**Conclusión.** **No hay años imposibles** (la columna de año está limpia: solo 2015–2017). Lo que sí aparece son unos pocos registros a sanear: una `adr` **negativa** (−6,4) y otra de **5400** (*outliers*/errores), del orden de **180** reservas **sin huéspedes** y **715** **sin noches**, y alguna fila con **ocupación absurda** (10 niños/bebés). En el pipeline descartamos las reservas sin huéspedes; las `adr` extremas podrían recortarse (*winsorizing*) si perjudicaran al modelo.

## 9. Filas duplicadas

Este dataset **no tiene identificador de reserva**, así que una "fila duplicada" es una reserva con **las 32 columnas idénticas** a otra. ¿Son errores de carga o reservas reales que casualmente coinciden? Es una decisión delicada, así que la investigamos a fondo.

In [26]:
# Filas que participan en algún grupo de duplicado EXACTO (las 32 columnas iguales)
dup_mask = df.duplicated(keep=False)
n_dup_filas = int(dup_mask.sum())
n_extra = int(df.duplicated().sum())  # filas que se borrarían dejando una por grupo
print(f"Filas en algún duplicado exacto: {n_dup_filas:,} ({n_dup_filas/len(df):.1%} del total)")
print(f"Filas 'extra' (se irían al deduplicar): {n_extra:,}  ->  quedarían {len(df)-n_extra:,} únicas")

Filas en algún duplicado exacto: 40,165 (33.6% del total)
Filas 'extra' (se irían al deduplicar): 31,994  ->  quedarían 87,396 únicas


### 9.1 ¿Errores o reservas reales? Tres pistas

In [27]:
# Pista 1: ¿hay conflictos de etiqueta? (mismas features, distinto is_canceled)
feat_cols = [c for c in df.columns if c != "is_canceled"]
g_lbl = df.groupby(feat_cols, dropna=False)["is_canceled"].nunique()
print("Grupos de features idénticas con etiqueta MIXTA (0 y 1):", int((g_lbl > 1).sum()))

Grupos de features idénticas con etiqueta MIXTA (0 y 1): 0


In [28]:
# Pista 2: ¿dónde se concentran? (% duplicado por segmento de mercado)
import numpy as np
tmp = df.assign(_dup=dup_mask)
por_seg = (tmp.groupby("market_segment")
              .agg(pct_duplicado=("_dup", "mean"), tasa_cancel=("is_canceled", "mean"), n=("_dup", "size"))
              .sort_values("pct_duplicado", ascending=False).round(3))
por_seg

,pct_duplicado,tasa_cancel,n
market_segment,,,
Groups,0.844,0.611,19811
Offline TA/TO,0.504,0.343,24219
Corporate,0.278,0.187,5295
Online TA,0.146,0.367,56477
Direct,0.111,0.153,12606
Complementary,0.101,0.131,743
Aviation,0.080,0.219,237
Undefined,0.000,1.000,2


In [29]:
# Pista 3: perfil de las filas duplicadas vs únicas
resumen_dup = pd.DataFrame({
    "duplicadas": [df.loc[dup_mask, "is_canceled"].mean(),
                   df.loc[dup_mask, "lead_time"].mean(),
                   df.loc[dup_mask, "total_of_special_requests"].mean()],
    "unicas":     [df.loc[~dup_mask, "is_canceled"].mean(),
                   df.loc[~dup_mask, "lead_time"].mean(),
                   df.loc[~dup_mask, "total_of_special_requests"].mean()],
}, index=["tasa_cancel", "lead_time_medio", "special_requests_medio"]).round(3)
resumen_dup

,duplicadas,unicas
tasa_cancel,0.584,0.262
lead_time_medio,160.082,75.585
special_requests_medio,0.287,0.716


### 9.2 El grupo de duplicados más grande

In [30]:
# El grupo idéntico más numeroso: ¿qué pinta tiene?
grupos = df.groupby(list(df.columns), dropna=False).size().sort_values(ascending=False)
print(f"El mayor grupo de filas idénticas tiene {int(grupos.iloc[0])} reservas.")
cols_muestra = ["hotel", "market_segment", "deposit_type", "customer_type",
                "lead_time", "arrival_date_month", "adr", "agent", "is_canceled"]
df[df.duplicated(keep=False)].value_counts(cols_muestra, dropna=False).head(5).to_frame("n")

El mayor grupo de filas idénticas tiene 180 reservas.


n
hotel      market_segment deposit_type customer_type lead_time arrival_date_month adr   agent is_canceled     
City Hotel Groups         Non Refund   Transient     277       November           100.0 NaN   1            180
                                                     68        February           75.0  37.0  1            150
           Offline TA/TO  Non Refund   Transient     34        December           90.0  19.0  1            140
                                                     188       June               130.0 119.0 1            109
           Groups         Non Refund   Transient     158       May                130.0 37.0  1            101

**Lectura — son, casi con certeza, reservas reales (no errores):**

- **Cero conflictos de etiqueta**: ningún grupo de features idénticas tiene a la vez `is_canceled=0` y `1`. Si fueran fallos de carga aleatorios, esperaríamos contradicciones; en cambio son perfectamente coherentes.
- **Se concentran donde hay reservas en bloque**: `Non Refund` ≈ 98 % duplicado, segmento `Groups` ≈ 84 %, `Transient-Party` ≈ 66 %. El mayor grupo son **180 filas idénticas** (City, *Groups*, *Non Refund*, 277 días de antelación, sin peticiones especiales, todas canceladas).
- **Su perfil grita "bloqueo especulativo"**: las duplicadas tienen **mucha más antelación** (lead_time ≈ 160 vs 76) y **menos peticiones especiales** (≈ 0,29 vs 0,72).

Sin un identificador de reserva, un turoperador que bloquea 180 habitaciones idénticas produce 180 filas idénticas que **son reservas distintas**, no copias. No podemos distinguir "bloque real" de "duplicado".

**Decisión: las MANTENEMOS** (no son errores; eliminarlas tiraría ~32.000 filas reales y cambiaría el problema). Pero hay un **riesgo a documentar** 👇

> ⚠️ **Riesgo de *fuga por duplicación* (train/test leakage).** Las filas duplicadas cancelan mucho más (**≈ 58 %** vs 26 % las únicas). Si las dejamos **y** partimos train/test al azar, filas idénticas caen en **ambos** conjuntos y el modelo puede *memorizarlas* → métricas de test **optimistas**. Es, probablemente, parte de por qué la ROC-AUC es tan alta (~0,96). Lo dejamos anotado como **limitación conocida**; una alternativa sería deduplicar *antes* de partir, a costa de descartar reservas reales. (Esto es EDA: el pipeline `src/` mantiene las filas.)

### 9.3 ¿Hay agencias detrás de los bloques especulativos?

Si los duplicados son **bloqueos especulativos**, quizá los genera un puñado de **agencias** (`agent`). Si fuera así, sería una **señal valiosa**: dejando que el modelo aprenda *qué agencia* hizo la reserva, aprende también su riesgo de cancelación. Vamos a verlo.

In [31]:
# agent viene como número (p. ej. 9.0); lo pasamos a etiqueta legible y marcamos los nulos
df_ag = df.copy()
df_ag["agent_label"] = df_ag["agent"].apply(
    lambda v: "Sin agencia" if pd.isna(v) else f"Agencia {int(float(v))}"
)
dup_mask = df_ag.duplicated(subset=[c for c in df.columns], keep=False)

# ¿Qué agencias acumulan más FILAS DUPLICADAS?
top_ag = (df_ag[dup_mask].groupby("agent_label")
          .agg(filas_dup=("is_canceled", "size"), tasa_cancel_dup=("is_canceled", "mean"))
          .sort_values("filas_dup", ascending=False).head(10))
top_ag["%_del_total_dup"] = (top_ag["filas_dup"] / int(dup_mask.sum()) * 100).round(1)
top_ag.round(3)

,filas_dup,tasa_cancel_dup,%_del_total_dup
agent_label,,,
Agencia 1,6611,0.788,16.5
Agencia 9,5396,0.523,13.4
Sin agencia,5220,0.529,13.0
Agencia 6,2420,0.380,6.0
Agencia 240,1547,0.586,3.9
Agencia 3,1108,0.653,2.8
Agencia 37,1082,0.638,2.7
Agencia 19,959,0.797,2.4
Agencia 21,765,0.635,1.9


In [32]:
# Concentración: ¿está el problema en pocas agencias?
dup_por_ag = df_ag[dup_mask].groupby("agent_label").size().sort_values(ascending=False)
tot_dup = int(dup_mask.sum())
for k in (1, 3, 5, 10):
    print(f"Top {k:>2} agencias  ->  {dup_por_ag.head(k).sum()/tot_dup*100:5.1f}% de todas las filas duplicadas")

Top  1 agencias  ->   16.5% de todas las filas duplicadas
Top  3 agencias  ->   42.9% de todas las filas duplicadas
Top  5 agencias  ->   52.8% de todas las filas duplicadas
Top 10 agencias  ->   64.2% de todas las filas duplicadas


In [33]:
# Perfil de riesgo POR agencia (entre las de cierto volumen): no todas son iguales
perfil = df_ag.assign(_dup=dup_mask).groupby("agent_label").agg(
    n_total=("is_canceled", "size"),
    tasa_cancel=("is_canceled", "mean"),
    pct_duplicado=("_dup", "mean"),
    lead_time_medio=("lead_time", "mean"),
    pct_groups=("market_segment", lambda s: (s == "Groups").mean()),
)
# Solo agencias con volumen suficiente para que el perfil sea fiable
perfil = perfil[perfil["n_total"] >= 500].sort_values("tasa_cancel", ascending=False).round(3)
perfil.head(12)

,n_total,tasa_cancel,pct_duplicado,lead_time_medio,pct_groups
agent_label,,,,,
Agencia 29,683,0.799,0.934,136.665,0.994
Agencia 19,1061,0.735,0.904,142.766,0.202
Agencia 1,7191,0.734,0.919,243.931,0.999
Agencia 20,540,0.665,0.874,153.565,0.135
Agencia 229,786,0.616,0.837,394.807,0.520
Agencia 37,1230,0.583,0.880,121.487,0.998
Agencia 21,875,0.578,0.874,218.296,0.037
Agencia 3,1336,0.577,0.829,150.767,0.142
Agencia 12,578,0.526,0.874,171.033,0.048


In [34]:
# Visual: tasa de cancelación vs % duplicado por agencia (tamaño = volumen)
perfil_plot = perfil.reset_index()
fig = px.scatter(
    perfil_plot, x="pct_duplicado", y="tasa_cancel",
    size="n_total", color="tasa_cancel", color_continuous_scale="Reds",
    hover_name="agent_label", size_max=45,
    labels={"pct_duplicado": "% de sus reservas que son duplicado",
            "tasa_cancel": "Tasa de cancelación", "n_total": "Nº reservas"},
    title="Agencias (≥500 reservas): riesgo de cancelación vs % duplicado",
)
fig.show()

**Lectura — sí, pero con matices (cada agencia tiene su perfil de riesgo).** Tu intuición acierta: hay una **agencia claramente especulativa**, pero no todas se comportan igual.

- 🚩 **Agencia 1** es el caso de libro: ~7.200 reservas, **el 92 % son bloques duplicados**, **73 % cancela**, antelación media de **244 días** y **100 % segmento *Groups***. Bloquea cupos en masa y cancela la mayoría.
- 🚩 Otras pequeñas y **muy tóxicas**: agencias 19, 29, 37 (~80 % de cancelación, casi todo duplicado).
- ✅ Pero **Agencia 9** —la de **más volumen** (~32.000 reservas)— solo tiene un 17 % de duplicados y cancela un **41 %** normalito: **volumen, no toxicidad**. Y la **Agencia 14** apenas cancela (**18 %**): fiable.
- Las duplicaciones están **concentradas**: las **3 primeras agencias acumulan ~43 %** y las **10 primeras ~64 %** de todas las filas duplicadas.

**Conclusión.** El nulo/identidad de `agent` **no es ruido**: codifica el **riesgo propio de cada agencia**. Por eso **conviene conservar `agent`** como categórica (con cardinalidad acotada) — así el modelo aprende que *Agencia 1 ≈ alto riesgo* y *Agencia 14 ≈ bajo*, justo lo que pedías. Matiza la idea anterior de `has_agent`: el **ID concreto** informa más que el simple "tiene/no tiene agencia".

## 10. Fugas de información (*data leakage*)

`reservation_status` y `reservation_status_date` describen lo que pasó **después** de decidir la cancelación. Si las dejáramos, el modelo "vería la respuesta". Lo demostramos con una tabla cruzada.

In [35]:
# Tabla cruzada: reservation_status frente a is_canceled
pd.crosstab(df["reservation_status"], df["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


**Confirmado.** `reservation_status = 'Canceled' / 'No-Show'` coincide **exactamente** con `is_canceled = 1`, y `'Check-Out'` con `0`: correspondencia perfecta. Dejar esa columna daría un acierto artificial de ~100 %, engañoso e inútil porque **no existe en el momento de predecir**. Descartamos `reservation_status` y `reservation_status_date`.

## 11. Variables numéricas

Repasamos **todas** las numéricas, una por una, mirando siempre cómo se relaciona cada una con la cancelación y **siempre por hotel** (porque ya vimos que City y Resort son casi dos negocios distintos). Para las **continuas** usamos una *caja* (box plot) más una **línea de tasa de cancelación por tramos** (binned); para las **discretas** (conteos, banderas 0/1) la tasa de cancelación por valor. El objetivo es ver qué variables separan de verdad a quien cancela de quien no.

In [36]:
# Helpers de esta sección: relación de cada variable con la cancelación, SIEMPRE por hotel.
def tasa_numerica(col, bins=None, etiquetas=None):
    """Tasa de cancelación de `col` por hotel.

    Si `bins` se indica, agrupa la numérica en intervalos; si no, usa el valor tal cual
    (para columnas discretas). Devuelve un DataFrame tidy y dibuja una línea por hotel.
    """
    d = df.copy()
    x = col
    if bins is not None:
        d["_bin"] = pd.cut(d[col], bins=bins, labels=etiquetas)
        x = "_bin"
    g = (d.groupby([x, "hotel"], observed=True)["is_canceled"]
           .agg(tasa_cancelacion="mean", reservas="size").reset_index())
    fig = px.line(g, x=x, y="tasa_cancelacion", color="hotel", markers=True,
                  color_discrete_map=COLOR_HOTEL, hover_data=["reservas"],
                  title=f"Tasa de cancelación según {col} (por hotel)",
                  labels={x: col, "tasa_cancelacion": "Tasa de cancelación", "hotel": "Hotel"})
    fig.update_yaxes(range=[0, 1], tickformat=".0%")
    fig.show()
    return g

def tasa_categorica_top(col, top=25, min_n=50):
    """Tasa de cancelación por categoría de `col`, ORDENADA de mayor a menor.

    Para alta cardinalidad: se queda con las `top` categorías de MAYOR cancelación
    entre las que tienen al menos `min_n` reservas (evita falsos 100% de muestras
    minúsculas). Dibuja barras horizontales coloreadas por tasa.
    """
    base = (df.groupby(col, observed=True)["is_canceled"]
              .agg(tasa_cancelacion="mean", reservas="size").reset_index())
    base = base[base["reservas"] >= min_n]
    base = base.sort_values("tasa_cancelacion", ascending=False).head(top)
    orden = base.sort_values("tasa_cancelacion").copy()  # asc para barh de mayor arriba
    # agent/company son float (9.0, 1.0); los pasamos a entero-string ("9", "1") para
    # que el eje no muestre ".0". Las columnas de texto (country) quedan igual.
    if pd.api.types.is_numeric_dtype(orden[col]):
        orden[col] = orden[col].astype("Int64").astype(str)
    else:
        orden[col] = orden[col].astype(str)
    fig = px.bar(orden, x="tasa_cancelacion", y=col, orientation="h",
                 color="tasa_cancelacion", color_continuous_scale="OrRd",
                 hover_data=["reservas"], text="tasa_cancelacion",
                 title=f"Tasa de cancelación por {col} (top {top}, ≥{min_n} reservas)",
                 labels={"tasa_cancelacion": "Tasa de cancelación", col: col})
    fig.update_traces(texttemplate="%{text:.0%}")
    fig.update_xaxes(range=[0, 1], tickformat=".0%")
    # Forzamos eje categórico: aunque las etiquetas sean números-como-texto ("9"),
    # queremos UNA barra por categoría, no una escala continua 0..500.
    fig.update_yaxes(type="category")
    fig.update_layout(height=max(400, 22*len(orden)))
    fig.show()
    return base

def tasa_categorica_hotel(col):
    """Tasa de cancelación por categoría de `col` desglosada por hotel (baja cardinalidad),
    ordenada por la tasa global de la categoría."""
    orden = (df.groupby(col, observed=True)["is_canceled"].mean()
               .sort_values(ascending=False).index.tolist())
    g = (df.groupby([col, "hotel"], observed=True)["is_canceled"]
           .agg(tasa_cancelacion="mean", reservas="size").reset_index())
    fig = px.bar(g, x=col, y="tasa_cancelacion", color="hotel", barmode="group",
                 category_orders={col: orden}, color_discrete_map=COLOR_HOTEL,
                 hover_data=["reservas"], text="tasa_cancelacion",
                 title=f"Tasa de cancelación por {col} y hotel",
                 labels={col: col, "tasa_cancelacion": "Tasa de cancelación", "hotel": "Hotel"})
    fig.update_traces(texttemplate="%{text:.0%}")
    fig.update_yaxes(range=[0, 1.05], tickformat=".0%")
    fig.show()
    return g

# Lista de columnas numéricas relevantes (incluimos is_canceled para el ranking de correlación)
NUM = [
    "lead_time", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "noches",
    "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes",
    "days_in_waiting_list", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "is_canceled",
]
NUM = [c for c in NUM if c in df.columns]
df[NUM].describe().transpose()

,count,mean,std,min,25%,50%,75%,max
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
noches,119390.0,3.427900,2.557439,0.00,2.00,3.000,4.0,69.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0
is_repeated_guest,119390.0,0.031912,0.175767,0.00,0.00,0.000,0.0,1.0


In [37]:
# Matriz de correlación entre numéricas + ranking frente a la clase
plot_matriz_correlacion(df[NUM], "is_canceled")


Correlaciones con 'is_canceled':
is_canceled                       1.000000
lead_time                         0.293123
previous_cancellations            0.110133
adults                            0.060017
days_in_waiting_list              0.054186
adr                               0.047557
stays_in_week_nights              0.024765
noches                            0.017779
arrival_date_week_number          0.008148
children                          0.005048
stays_in_weekend_nights          -0.001791
arrival_date_day_of_month        -0.006130
babies                           -0.032491
previous_bookings_not_canceled   -0.057358
is_repeated_guest                -0.084793
booking_changes                  -0.144381
required_car_parking_spaces      -0.195498
total_of_special_requests        -0.234658
Name: is_canceled, dtype: float64


**Lectura de correlaciones.** No hay parejas fuertemente redundantes (salvo la trivial entre `noches` y sus dos sumandos). La numérica que más correlaciona con la cancelación es `lead_time` (positiva: más antelación → más cancela); `total_of_special_requests` y `required_car_parking_spaces` van en sentido contrario. Las correlaciones lineales son moderadas, pero eso **no** significa que las variables sean inútiles: muchas relacionan de forma **no lineal** o **escalonada** con la clase, y eso se ve mejor mirando la **tasa de cancelación por tramos**, que es justo lo que hacemos a continuación variable a variable.

### 11.1 `lead_time` (antelación)

El predictor numérico más fuerte. La caja muestra que las reservas canceladas se hacen con **mucha más antelación**; la línea por tramos lo confirma con un patrón **monótono creciente**: de **~10 %** (0-7 días) a **~68 %** (más de un año).

In [38]:
# lead_time por clase y por hotel
fig = px.box(
    df,
    x="hotel",
    y="lead_time",
    color="is_canceled",
    title="Antelación (lead_time) por cancelación y hotel",
    labels={"lead_time": "Días de antelación", "hotel": "Hotel", "is_canceled": "Cancelada"},
    color_discrete_map={0: "#2c7fb8", 1: "#d95f02"},
    width=800, height=450,
)
fig.show()

In [39]:
tasa_numerica("lead_time", bins=[-1, 7, 30, 90, 180, 365, 10000],
              etiquetas=["0-7", "8-30", "31-90", "91-180", "181-365", "365+"])

,_bin,hotel,tasa_cancelacion,reservas
0,0-7,City Hotel,0.122039,10808
1,0-7,Resort Hotel,0.065227,8938
2,8-30,City Hotel,0.309065,12554
3,8-30,Resort Hotel,0.219013,6406
4,31-90,City Hotel,0.399096,20797
5,31-90,Resort Hotel,0.324463,8756
6,91-180,City Hotel,0.479943,18223
7,91-180,Resort Hotel,0.374270,8216
8,181-365,City Hotel,0.627211,14244
9,181-365,Resort Hotel,0.412740,7300


**Lectura.** Más antelación → más cancelación, casi linealmente por tramos: 0-7 ≈ 10 %, 8-30 ≈ 28 %, 31-90 ≈ 38 %, 91-180 ≈ 45 %, 181-365 ≈ 56 % y 365+ ≈ 68 %. Es **el predictor numérico más fuerte** del dataset.

### 11.2 `adr` (precio medio diario)

A diferencia de `lead_time`, el precio **apenas separa por sí solo**: la tasa por tramos es bastante plana (~35-39 %). Lo que sí cambia mucho son los **rangos de precio entre hoteles**.

In [40]:
# Distribución de adr por hotel y clase (recortamos outliers extremos: adr < 400)
df_adr = df[(df["adr"] >= 0) & (df["adr"] < 400)]
fig = px.box(
    df_adr,
    x="hotel",
    y="adr",
    color="is_canceled",
    title="Precio medio diario (adr) por hotel y cancelación (recortado a adr<400)",
    labels={"adr": "ADR (precio medio diario)", "hotel": "Hotel", "is_canceled": "Cancelada"},
    color_discrete_map={0: "#2c7fb8", 1: "#d95f02"},
    width=800, height=450,
)
fig.show()

In [41]:
# Tramos de adr por cuantiles aproximados (recortamos la cola extrema)
tasa_numerica("adr", bins=[-1, 64, 85, 105, 135, 400],
              etiquetas=["≤64", "64-85", "85-105", "105-135", "135-400"])

,_bin,hotel,tasa_cancelacion,reservas
0,≤64,City Hotel,0.578555,8459
1,≤64,Resort Hotel,0.224372,15452
2,64-85,City Hotel,0.363047,16659
3,64-85,Resort Hotel,0.297685,7817
4,85-105,City Hotel,0.419367,19353
5,85-105,Resort Hotel,0.263694,3925
6,105-135,City Hotel,0.410962,20216
7,105-135,Resort Hotel,0.298035,4174
8,135-400,City Hotel,0.391735,14640
9,135-400,Resort Hotel,0.350794,8686


**Lectura.** La tasa de cancelación es **plana** (~35-39 %) a lo largo de los tramos de precio: `adr` apenas discrimina por sí sola. Su interés es sobre todo **descriptivo** (los dos hoteles juegan en rangos de precio distintos), no tanto predictivo de forma directa.

### 11.3 `stays_in_week_nights` (noches entre semana)

Conteo discreto: la tasa **sube hasta las 2 noches** (~44 %) y luego baja un poco. Relación **poco monótona**.

In [42]:
tasa_numerica("stays_in_week_nights", bins=[-1, 0, 1, 2, 3, 5, 100],
              etiquetas=["0", "1", "2", "3", "4-5", "6+"])

,_bin,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.298207,4963
1,0,Resort Hotel,0.161447,2682
2,1,City Hotel,0.376233,21088
3,1,Resort Hotel,0.205704,9222
4,2,City Hotel,0.478203,26403
5,2,Resort Hotel,0.308749,7281
6,3,City Hotel,0.409077,16371
7,3,Resort Hotel,0.307117,5887
8,4-5,City Hotel,0.398746,9407
9,4-5,Resort Hotel,0.320929,11233


### 11.4 `stays_in_weekend_nights` (noches de fin de semana)

Casi **plana** (~36-38 %): el número de noches de fin de semana apenas aporta señal.

In [43]:
tasa_numerica("stays_in_weekend_nights", bins=[-1, 0, 1, 2, 100],
              etiquetas=["0", "1", "2", "3+"])

,_bin,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.429860,37817
1,0,Resort Hotel,0.228827,14181
2,1,City Hotel,0.399226,21434
3,1,Resort Hotel,0.266536,9192
4,2,City Hotel,0.406662,19333
5,2,Resort Hotel,0.330805,13975
6,3+,City Hotel,0.572386,746
7,3+,Resort Hotel,0.296460,2712


### 11.5 `noches` (derivada = semana + fin de semana)

Es la **suma** de las dos anteriores (`stays_in_week_nights` + `stays_in_weekend_nights`). La miramos por tramos para ver la estancia total; como cabía esperar de sus componentes, separa poco.

In [44]:
tasa_numerica("noches", bins=[-1, 1, 3, 7, 100],
              etiquetas=["1", "2-3", "4-7", "8+"])

,_bin,hotel,tasa_cancelacion,reservas
0,1,City Hotel,0.303315,13603
1,1,Resort Hotel,0.145106,8132
2,2-3,City Hotel,0.465344,42807
3,2-3,Resort Hotel,0.301713,11912
4,4-7,City Hotel,0.386674,21672
5,4-7,Resort Hotel,0.321422,16007
6,8+,City Hotel,0.541667,1248
7,8+,Resort Hotel,0.300075,4009


### 11.6 `adults` (adultos)

Las reservas de **0-1 adulto** cancelan menos (~29 %); a partir de 2 adultos sube a ~39 % y se mantiene alta (3+ ≈ 35-41 %).

In [45]:
tasa_numerica("adults", bins=[-1, 1, 2, 100],
              etiquetas=["0-1", "2", "3+"])

,_bin,hotel,tasa_cancelacion,reservas
0,0-1,City Hotel,0.341816,16269
1,0-1,Resort Hotel,0.170647,7161
2,2,City Hotel,0.442125,58255
3,2,Resort Hotel,0.302371,31425
4,3+,City Hotel,0.371411,4806
5,3+,Resort Hotel,0.270014,1474


### 11.7 `children` (niños)

Variable discreta (apenas 4 nulos en todo el dataset). Con 0 niños se cancela ~37 %; con 1 o más sube algo (~32-42 % según el conteo). Aporta poco por sí sola.

In [46]:
tasa_numerica("children")

,children,hotel,tasa_cancelacion,reservas
0,0.0,City Hotel,0.421531,74220
1,0.0,Resort Hotel,0.268154,36576
2,1.0,City Hotel,0.333774,3023
3,1.0,Resort Hotel,0.303591,1838
4,2.0,City Hotel,0.389328,2024
5,2.0,Resort Hotel,0.462531,1628
6,3.0,City Hotel,0.254237,59
7,3.0,Resort Hotel,0.117647,17
8,10.0,Resort Hotel,1.000000,1


### 11.8 `babies` (bebés)

El **99,2 %** de las reservas tienen 0 bebés, así que la tasa es casi constante y la variable es **poco informativa** (demasiado concentrada en un único valor).

In [47]:
tasa_numerica("babies")

,babies,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.418409,78961
1,0,Resort Hotel,0.278877,39512
2,1,City Hotel,0.177285,361
3,1,Resort Hotel,0.187384,539
4,2,City Hotel,0.000000,6
5,2,Resort Hotel,0.222222,9
6,9,City Hotel,0.000000,1
7,10,City Hotel,0.000000,1


### 11.9 `is_repeated_guest` (cliente recurrente)

Bandera 0/1 con una señal clara: los **clientes recurrentes cancelan mucho menos**. Quien no es recurrente cancela **37,8 %**; quien sí lo es, solo **14,5 %**.

In [48]:
tasa_numerica("is_repeated_guest")

,is_repeated_guest,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.422534,77298
1,0,Resort Hotel,0.287629,38282
2,1,City Hotel,0.217028,2032
3,1,Resort Hotel,0.062430,1778


### 11.10 `previous_cancellations` (cancelaciones previas)

**Una de las señales más potentes del dataset.** Con 0 cancelaciones previas se cancela **33,9 %**; con **una o más**, ¡un **91,6 %**! Quien ya canceló antes vuelve a cancelar casi siempre.

In [49]:
tasa_numerica("previous_cancellations", bins=[-1, 0, 100],
              etiquetas=["0", "≥1"])

,_bin,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.379816,73941
1,0,Resort Hotel,0.261722,38965
2,≥1,City Hotel,0.931156,5389
3,≥1,Resort Hotel,0.843836,1095


### 11.11 `previous_bookings_not_canceled` (reservas previas NO canceladas)

El reverso de la anterior: un **historial fiable** protege. Con 0 reservas previas cumplidas se cancela ~38 %; con **una o más**, solo **~5 %**.

In [50]:
tasa_numerica("previous_bookings_not_canceled", bins=[-1, 0, 100],
              etiquetas=["0", "≥1"])

,_bin,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.424237,77742
1,0,Resort Hotel,0.290391,38028
2,≥1,City Hotel,0.076196,1588
3,≥1,Resort Hotel,0.038878,2032


### 11.12 `booking_changes` (cambios en la reserva)

Modificar la reserva es un signo de **compromiso** y reduce la cancelación: 0 cambios → **40,9 %**, 1 cambio → **14,2 %**, 2 o más → **~18 %**.

In [51]:
tasa_numerica("booking_changes", bins=[-1, 0, 1, 100],
              etiquetas=["0", "1", "2+"])

,_bin,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.454997,69062
1,0,Resort Hotel,0.309066,32252
2,1,City Hotel,0.137445,7232
3,1,Resort Hotel,0.148656,5469
4,2+,City Hotel,0.225626,3036
5,2+,Resort Hotel,0.145789,2339


### 11.13 `days_in_waiting_list` (días en lista de espera)

Pasar por **lista de espera dispara la cancelación**: 0 días (sin espera) → **36,2 %**, una o más → **~65 %**.

In [52]:
tasa_numerica("days_in_waiting_list", bins=[-1, 0, 100],
              etiquetas=["0 (sin espera)", "≥1"])

,_bin,hotel,tasa_cancelacion,reservas
0,0 (sin espera),City Hotel,0.405340,75887
1,0 (sin espera),Resort Hotel,0.278985,39805
2,≥1,City Hotel,0.702097,2813
3,≥1,Resort Hotel,0.086957,138


### 11.14 `required_car_parking_spaces` (plazas de parking) ⚠️ *sospechosa de fuga*

Quien **no** pide parking cancela **39,5 %**, pero quien pide **una o más plazas cancela 0,0 % EXACTO**: de **7.416** reservas con parking, **ninguna** se canceló. Una señal *perfecta* siempre debe hacernos sospechar de **fuga de información** (*leakage*), igual que con `reservation_status` (§10).

In [53]:
tasa_numerica("required_car_parking_spaces")

,required_car_parking_spaces,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.427652,77404
1,0,Resort Hotel,0.321724,34570
2,1,City Hotel,0.000000,1921
3,1,Resort Hotel,0.000000,5462
4,2,City Hotel,0.000000,3
5,2,Resort Hotel,0.000000,25
6,3,City Hotel,0.000000,2
7,3,Resort Hotel,0.000000,1
8,8,Resort Hotel,0.000000,2


**Prueba de control.** Si el 0 % se debiera a que quien pide parking además paga *no reembolsable*, el efecto desaparecería al fijar `deposit_type`. No es el caso: dentro de **`No Deposit`** (reservas que sí se cancelan, ~30 %), las que piden parking siguen en **0 / 7.397** cancelaciones. Eso no es *predictivo*, es **determinista** → el patrón clásico de una variable que se rellena **después** del hecho.

In [54]:
# Control: ¿sigue siendo 0% dentro de las reservas cancelables (No Deposit)?
nd = df[df["deposit_type"] == "No Deposit"]
print("Dentro de 'No Deposit' (tasa global %.1f%%):" % (nd["is_canceled"].mean() * 100))
print("  sin parking: %.4f" % nd.loc[nd["required_car_parking_spaces"] == 0, "is_canceled"].mean())
con = nd[nd["required_car_parking_spaces"] > 0]
print("  con parking: %.4f  (n=%d, cancelaciones=%d)" % (
    con["is_canceled"].mean(), len(con), int(con["is_canceled"].sum())))

Dentro de 'No Deposit' (tasa global 28.4%):
  sin parking: 0.3054
  con parking: 0.0000  (n=7397, cancelaciones=0)


**Conclusión — candidata a eliminar.** La plaza de parking se asigna/registra al **llegar al hotel** (check-in): si el cliente llegó, ya **no** va a cancelar. Es información del **futuro** que no existe cuando queremos predecir, y da una falsa sensación de seguridad ("esta reserva no se cancelará"). Se comporta como un `reservation_status` suave. **Recomendación: tratarla como fuga y excluirla** del modelo. *(Decisión de EDA; el pipeline `src/` todavía la incluye — pendiente de quitar, igual que las features `has_*`.)*

### 11.15 `total_of_special_requests` (peticiones especiales)

**Monótona decreciente:** cada petición especial es una señal de compromiso. 0 peticiones → **47,7 %**, 1 → ~22 %, 2 → ~22 %, 3 → ~18 %, 4 → ~11 %, 5 → ~5 %. A más peticiones, menos cancelación.

In [55]:
tasa_numerica("total_of_special_requests")

,total_of_special_requests,hotel,tasa_cancelacion,reservas
0,0,City Hotel,0.549242,47957
1,0,Resort Hotel,0.322705,22361
2,1,City Hotel,0.220401,21420
3,1,Resort Hotel,0.219973,11806
4,2,City Hotel,0.213584,8142
5,2,Resort Hotel,0.233478,4827
6,3,City Hotel,0.176434,1587
7,3,Resort Hotel,0.182418,910
8,4,City Hotel,0.106061,198
9,4,Resort Hotel,0.105634,142


### 11.16 `arrival_date_day_of_month` (día del mes de llegada)

**Plana** (~37 % en los tres tercios del mes): el día concreto del mes no aporta señal.

In [56]:
tasa_numerica("arrival_date_day_of_month", bins=[0, 10, 20, 31],
              etiquetas=["1-10", "11-20", "21-31"])

,_bin,hotel,tasa_cancelacion,reservas
0,1-10,City Hotel,0.419646,25705
1,1-10,Resort Hotel,0.285978,13001
2,11-20,City Hotel,0.423505,26786
3,11-20,Resort Hotel,0.266515,13230
4,21-31,City Hotel,0.408771,26839
5,21-31,Resort Hotel,0.280425,13829


### 11.17 `arrival_date_week_number` y `arrival_date_year`

No las repetimos aquí: la **semana del año** (estacionalidad) ya se analizó en §7 y el **año de llegada** (cobertura temporal parcial y confundida con la estación) en §6.

**Resumen numérico.** Las numéricas que más separan a quien cancela son:

- `previous_cancellations`: **33,9 % → 91,6 %** (la más brutal: quien ya canceló, repite).
- `lead_time`: **monótona 10 % → 68 %** (más antelación, más cancelación).
- `total_of_special_requests`: **decreciente** (más peticiones = más compromiso).
- `is_repeated_guest` (37,8 % → 14,5 %), `previous_bookings_not_canceled` (38 % → ~5 %) y `days_in_waiting_list` (36 % → ~65 %) también dan señal clara.

⚠️ **`required_car_parking_spaces` queda FUERA de la lista a propósito**: separa de forma perfecta (39,5 % → 0 %) pero es **sospechosa de fuga** —se conoce en el check-in (§11.14)— y no debería usarse como predictor.

El resto (`adr`, noches, `adults`, `children`, `babies`, día del mes) aportan **poco por sí solos**, aunque pueden sumar en combinación dentro de un modelo no lineal.

## 12. Variables categóricas

Exploramos **cada** categórica con su tasa de cancelación, siempre **ordenada de mayor a menor** y, en baja cardinalidad, **desglosada por hotel**. En alta cardinalidad (`country`, `agent`, `company`) nos quedamos con el **TOP por cancelación** entre las categorías con suficientes reservas (≥50, para evitar falsos 100 % de muestras minúsculas).

⚠️ **Aclaración importante:** el pipeline de `src/` recorta las categóricas de alta cardinalidad por las **más frecuentes** (no por las que más cancelan). Aquí ordenamos por **cancelación** porque el objetivo es **entender** la señal, no replicar el *one-hot encoding* del pipeline.

### 12.1 `deposit_type` (tipo de depósito)

**La variable más predictiva del dataset.** `Non Refund` (no reembolsable) cancela ~**99 %** (≈ 98 % en City, ≈ 96 % en Resort) — un *proxy* de bloqueos especulativos y *no-shows*. `No Deposit` ronda el **28 %** y `Refundable` es bajo.

In [57]:
tasa_categorica_hotel("deposit_type")

,deposit_type,hotel,tasa_cancelacion,reservas
0,No Deposit,City Hotel,0.304687,66442
1,No Deposit,Resort Hotel,0.247389,38199
2,Non Refund,City Hotel,0.998135,12868
3,Non Refund,Resort Hotel,0.959860,1719
4,Refundable,City Hotel,0.700000,20
5,Refundable,Resort Hotel,0.154930,142


### 12.2 `market_segment` (segmento de mercado)

`Groups` cancela ~**61 %** global y **muy distinto por hotel** (City ≈ 69 % vs Resort ≈ 42 %): otro motivo para mirar por hotel. (Hay un `Undefined` con tasa 100 %, pero son solo 2 filas: anecdótico.)

In [58]:
tasa_categorica_hotel("market_segment")

,market_segment,hotel,tasa_cancelacion,reservas
0,Aviation,City Hotel,0.219409,237
1,Complementary,City Hotel,0.118081,542
2,Complementary,Resort Hotel,0.164179,201
3,Corporate,City Hotel,0.214668,2986
4,Corporate,Resort Hotel,0.152014,2309
5,Direct,City Hotel,0.173314,6093
6,Direct,Resort Hotel,0.134807,6513
7,Groups,City Hotel,0.688587,13975
8,Groups,Resort Hotel,0.423920,5836
9,Offline TA/TO,City Hotel,0.428316,16747


### 12.3 `customer_type` (tipo de cliente)

`Transient` es el que **más cancela** (~41 %); `Group` el que **menos** (~10 %).

In [59]:
tasa_categorica_hotel("customer_type")

,customer_type,hotel,tasa_cancelacion,reservas
0,Contract,City Hotel,0.480435,2300
1,Contract,Resort Hotel,0.088401,1776
2,Group,City Hotel,0.098976,293
3,Group,Resort Hotel,0.105634,284
4,Transient,City Hotel,0.456165,59404
5,Transient,Resort Hotel,0.311695,30209
6,Transient-Party,City Hotel,0.280967,17333
7,Transient-Party,Resort Hotel,0.194969,7791


### 12.4 `distribution_channel` (canal de distribución)

`TA/TO` (agencias/turoperadores) es el canal con **más cancelación**; `Direct` y `Corporate` son bajos.

In [60]:
tasa_categorica_hotel("distribution_channel")

,distribution_channel,hotel,tasa_cancelacion,reservas
0,Corporate,City Hotel,0.230634,3408
1,Corporate,Resort Hotel,0.210462,3269
2,Direct,City Hotel,0.181711,6780
3,Direct,Resort Hotel,0.168468,7865
4,GDS,City Hotel,0.191710,193
5,TA/TO,City Hotel,0.450257,68945
6,TA/TO,Resort Hotel,0.314918,28925
7,Undefined,City Hotel,1.000000,4
8,Undefined,Resort Hotel,0.000000,1


### 12.5 `hotel`

Ya analizado en §2 y §3: **City Hotel** cancela **41,7 %** frente al **Resort Hotel** con **27,8 %**. Lo recordamos con una barra simple.

In [61]:
# Tasa de cancelación por hotel (recordatorio de §2/§3)
hot = (df.groupby("hotel", observed=True)["is_canceled"]
         .agg(tasa_cancelacion="mean", reservas="size").reset_index()
         .sort_values("tasa_cancelacion", ascending=False))
fig = px.bar(hot, x="hotel", y="tasa_cancelacion", color="hotel",
             color_discrete_map=COLOR_HOTEL, text="tasa_cancelacion",
             hover_data=["reservas"],
             title="Tasa de cancelación por hotel",
             labels={"hotel": "Hotel", "tasa_cancelacion": "Tasa de cancelación"})
fig.update_traces(texttemplate="%{text:.1%}")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()
hot

,hotel,tasa_cancelacion,reservas
0,City Hotel,0.417270,79330
1,Resort Hotel,0.277634,40060


### 12.6 `meal` (régimen de comidas)

In [62]:
tasa_categorica_hotel("meal")

,meal,hotel,tasa_cancelacion,reservas
0,BB,City Hotel,0.428007,62305
1,BB,Resort Hotel,0.261390,30005
2,FB,City Hotel,0.795455,44
3,FB,Resort Hotel,0.587533,754
4,HB,City Hotel,0.379772,6417
5,HB,Resort Hotel,0.316555,8046
6,SC,City Hotel,0.375142,10564
7,SC,Resort Hotel,0.034884,86
8,Undefined,Resort Hotel,0.244654,1169


### 12.7 `reserved_room_type` (tipo de habitación reservada)

Diez categorías de baja cardinalidad: las miramos por hotel directamente.

In [63]:
tasa_categorica_hotel("reserved_room_type")

,reserved_room_type,hotel,tasa_cancelacion,reservas
0,A,City Hotel,0.435306,62595
1,A,Resort Hotel,0.272747,23399
2,B,City Hotel,0.330045,1115
3,B,Resort Hotel,0.000000,3
4,C,City Hotel,0.357143,14
5,C,Resort Hotel,0.330065,918
6,D,City Hotel,0.352396,11768
7,D,Resort Hotel,0.263016,7433
8,E,City Hotel,0.325177,1553
9,E,Resort Hotel,0.282818,4982


### 12.8 `assigned_room_type` (tipo de habitación asignada) → **FUGA: se elimina**

🚫 **Decisión: eliminamos `assigned_room_type` por fuga de datos** (misma familia que `required_car_parking_spaces`, §11). El tipo **asignado** se conoce en el **check-in**, no en el momento de la reserva, así que como predictor filtra información del futuro. La prueba está en la relación **reservada vs asignada**: reasignar habitación implica que el cliente **se presentó**, de modo que un valor asignado distinto del reservado es casi un sello de "**no canceló**".

In [64]:
# Evidencia de la fuga: ¿coincide la habitación reservada con la asignada?
coincide = df["reserved_room_type"] == df["assigned_room_type"]
print(f"reservada == asignada : {coincide.sum():>6}  | tasa cancelación {df.loc[coincide, 'is_canceled'].mean():.1%}")
print(f"reservada != asignada : {(~coincide).sum():>6}  | tasa cancelación {df.loc[~coincide, 'is_canceled'].mean():.1%}")
print()
for v, etiqueta in [(0, "NO cancelada"), (1, "cancelada")]:
    sub = df[df["is_canceled"] == v]
    difiere = (sub["reserved_room_type"] != sub["assigned_room_type"]).mean()
    print(f"{etiqueta:13s}: la habitación difiere en {difiere:.1%} de los casos")

reservada == asignada : 104473  | tasa cancelación 41.6%
reservada != asignada :  14917  | tasa cancelación 5.4%

NO cancelada : la habitación difiere en 18.8% de los casos
cancelada    : la habitación difiere en 1.8% de los casos


La tabla por hotel se mantiene solo a título ilustrativo; la variable **no entra** en el modelo (`reserved_room_type`, que sí se conoce al reservar, **sí se conserva**).

In [65]:
tasa_categorica_hotel("assigned_room_type")

,assigned_room_type,hotel,tasa_cancelacion,reservas
0,A,City Hotel,0.471889,57007
1,A,Resort Hotel,0.354746,17046
2,B,City Hotel,0.250998,2004
3,B,Resort Hotel,0.056604,159
4,C,City Hotel,0.093168,161
5,C,Resort Hotel,0.194670,2214
6,D,City Hotel,0.285190,14983
7,D,Resort Hotel,0.202050,10339
8,E,City Hotel,0.249077,2168
9,E,Resort Hotel,0.253281,5638


### 12.9 `country` (país de origen)

Alta cardinalidad → top 25 por cancelación con ≥50 reservas. Entre los países de **gran volumen**, **Portugal (PRT)** —el mercado local— cancela ~**57 %**, frente a extranjeros con tasas mucho menores (GBR/FRA/DEU ≈ 17-20 %). El top-25 por tasa también arrastra países pequeños con tasas altas (de ahí el filtro `min_n=50`).

In [66]:
tasa_categorica_top("country", top=25, min_n=50)

,country,tasa_cancelacion,reservas
5,ARE,0.843137,51
135,PRT,0.566351,48590
1,AGO,0.566298,362
31,CHN,0.462462,999
103,MAR,0.420849,259
91,KOR,0.413534,133
162,TUR,0.411290,248
174,ZAF,0.387500,80
100,LUX,0.379791,287
140,RUS,0.378165,632


### 12.10 `agent` (agencia)

Los IDs de agencia son números (ver §5 y §9.3). En **§9.3** ya vimos que algunas agencias —como la **1**— cancelan ~73 % por sus bloqueos especulativos. Aquí mostramos solo el **top por tasa** entre las de cierto volumen.

In [67]:
tasa_categorica_top("agent", top=25, min_n=50)

,agent,tasa_cancelacion,reservas
39,41.0,1.000000,75
172,236.0,1.000000,247
135,170.0,1.000000,93
233,326.0,0.975758,165
29,31.0,0.950617,162
51,58.0,0.880597,335
323,495.0,0.877193,57
27,29.0,0.799414,683
41,44.0,0.794521,292
59,68.0,0.781991,211


### 12.11 `company` (empresa)

⚠️ `company` tiene **94 % de nulos** y grupos muy pequeños, así que el ranking por empresa es **ruidoso** (pocas reservas por ID). Usamos `min_n=30` y lo tomamos con pinzas. La señal robusta de esta columna es la **binaria** `has_company` (ver §5): tener empresa **protege** contra la cancelación.

In [68]:
tasa_categorica_top("company", top=25, min_n=30)

,company,tasa_cancelacion,reservas
215,348.0,1.000000,59
246,385.0,1.000000,30
115,202.0,1.000000,38
113,197.0,0.744681,47
37,67.0,0.655431,267
187,308.0,0.363636,33
169,280.0,0.354167,48
28,51.0,0.353535,99
69,110.0,0.326923,52
81,135.0,0.287879,66


**Resumen categórico.** La señal más fuerte de todo el dataset es `deposit_type` (`Non Refund` cancela ~**99 %**). Le siguen, modulando el riesgo, `market_segment`, `customer_type` y `distribution_channel`. La **identidad** aporta a través de `country` (el mercado local **PRT** cancela mucho) y de `agent` (ver §9.3, donde se ve el riesgo propio de cada agencia). En cambio `assigned_room_type` se **descarta por fuga** (se conoce en el check-in, §12.8); `reserved_room_type` sí se conserva.

## 13. Reducción de cardinalidad de las categóricas

`agent` (333), `country` (177) y `company` (352) tienen **demasiadas categorías** para un *one-hot* directo: generan cientos de columnas, muchas con poquísimas muestras (ruido). La idea: **quedarnos con las categorías que dan señal clara** —las que cancelan **mucho** o **muy poco**— y **agrupar el resto** en `Otros`. Los **nulos** se tratan como categoría propia (en `company` los imputamos a **`no_company`** = *reserva sin empresa*, señal real según §5; en `agent`/`country` quedan como `Desconocido`). Así el modelo aprende "esta agencia/país/empresa es de riesgo (o muy fiable)" sin pagar el coste de cientos de *dummies*.

**Regla, con dos salvaguardas:**

1. **Soporte mínimo** `n >= 100` reservas: una categoría con 3 reservas y 100 % de cancelación es ruido, no señal.
2. **Umbrales adaptativos por variable:** tomamos el **máximo** de tasa de esa variable como referencia (=100 %) y guardamos las categorías con tasa **> 60 % del máximo** (alto riesgo) o **< 30 % del máximo** (muy fiables). Adaptar por variable evita que, p. ej., `country` —cuyo máximo es Portugal con ~57 %— se quede sin lado "alto".

Para cada variable mostramos **tres gráficos**: (a) cobertura de reservas por grupo, (b) cardinalidad antes/después y (c) el **detalle de las categorías conservadas** con su tasa de cancelación.

> ⚠️ **Cuidado con la fuga.** Elegir qué categorías conservar **mirando su tasa de cancelación usa la variable objetivo**. Si esa lista se calcula sobre **todo** el dataset y luego se evalúa, hay **fuga train/test** (métricas optimistas). Aquí, en el EDA, lo hacemos sobre todo el dataset **solo para ilustrar**. Al llevarlo a `src/` debe hacerse como una transformación **ajustada solo en train** (la lista `KEEP` se calcula con los datos de entrenamiento y se aplica a test).

In [69]:
# Helper: clasifica cada categoria en KEEP-alto / KEEP-bajo / Otros / (nulos) y dibuja
# (a) cobertura por grupo, (b) cardinalidad antes/despues y
# (c) el detalle de las categorias CONSERVADAS con su tasa de cancelacion.
def reducir_cardinalidad(col, hi_frac=0.60, lo_frac=0.30, min_n=100, null_label="Desconocido (nulo)"):
    """Reduccion supervisada de categorias de `col`.

    Conserva las categorias con >= min_n reservas cuya tasa de cancelacion supera
    hi_frac*max (alto riesgo) o es menor que lo_frac*max (muy fiables); el resto se
    agrupa en 'Otros'. Los nulos se imputan a una categoria propia `null_label`
    (p. ej. 'no_company' = reserva sin empresa), que siempre se conserva como columna.
    Dibuja cobertura, cardinalidad y el detalle de las categorias conservadas.
    """
    s = df[col]
    g = df.assign(_c=s).groupby("_c", observed=True)["is_canceled"].agg(rate="mean", n="size")
    enough = g[g["n"] >= min_n]
    max_rate = enough["rate"].max()
    hi_cut, lo_cut = hi_frac * max_rate, lo_frac * max_rate
    hi = set(enough[enough["rate"] > hi_cut].index)
    lo = set(enough[enough["rate"] < lo_cut].index)

    def bucket(v):
        if pd.isna(v):
            return null_label
        if v in hi:
            return "KEEP alto riesgo"
        if v in lo:
            return "KEEP baja cancelacion"
        return "Otros (agrupado)"

    resumen = (df.assign(_grupo=s.map(bucket))
                 .groupby("_grupo")["is_canceled"]
                 .agg(reservas="size", tasa_cancelacion="mean").reset_index())
    resumen["pct_reservas"] = resumen["reservas"] / len(df)

    orden = ["KEEP alto riesgo", "KEEP baja cancelacion", "Otros (agrupado)", null_label]
    colores = {"KEEP alto riesgo": "#d7301f", "KEEP baja cancelacion": "#2c7fb8",
               "Otros (agrupado)": "#999999", null_label: "#7570b3"}

    # (a) cobertura: % de reservas por grupo, etiquetado con su tasa de cancelacion
    fig = px.bar(resumen, x="_grupo", y="pct_reservas", color="_grupo",
                 category_orders={"_grupo": orden}, color_discrete_map=colores,
                 text=resumen["tasa_cancelacion"].map(lambda r: f"tasa {r:.0%}"),
                 title=f"{col}: cobertura de reservas por grupo (etiqueta = tasa de cancelacion)",
                 labels={"_grupo": "", "pct_reservas": "% de reservas"})
    fig.update_yaxes(tickformat=".0%")
    fig.update_traces(textposition="outside")
    fig.show()

    # (b) cardinalidad antes vs despues
    n_keep = len(hi) + len(lo)
    extra = 1 + int(bool(s.isna().any()))  # 'Otros' siempre; null_label si hay nulos
    card = pd.DataFrame({"estado": ["Antes (one-hot directo)", "Despues (reducido)"],
                         "columnas": [int(g.shape[0]), n_keep + extra]})
    fig2 = px.bar(card, x="estado", y="columnas", color="estado", text="columnas",
                  color_discrete_sequence=["#bdbdbd", "#1b9e77"],
                  title=f"{col}: numero de categorias antes vs despues",
                  labels={"estado": "", "columnas": "Nº de columnas one-hot"})
    fig2.update_traces(textposition="outside")
    fig2.show()

    # (c) detalle: cada categoria CONSERVADA y su tasa de cancelacion (incl. la de nulos)
    det = g.loc[list(hi | lo)].reset_index()
    det.columns = [col, "rate", "n"]
    det["grupo"] = det[col].map(lambda v: "KEEP alto riesgo" if v in hi else "KEEP baja cancelacion")
    if pd.api.types.is_numeric_dtype(det[col]):
        det[col] = det[col].astype("Int64").astype(str)
    else:
        det[col] = det[col].astype(str)
    if s.isna().any():
        det = pd.concat([det, pd.DataFrame([{col: null_label,
                                             "rate": df.loc[s.isna(), "is_canceled"].mean(),
                                             "n": int(s.isna().sum()), "grupo": null_label}])],
                        ignore_index=True)
    det = det.sort_values("rate")
    fig3 = px.bar(det, x="rate", y=col, color="grupo", orientation="h",
                  category_orders={"grupo": orden}, color_discrete_map=colores,
                  text=det["rate"].map(lambda r: f"{r:.0%}"), hover_data={"n": True},
                  title=f"{col}: categorias CONSERVADAS y su tasa de cancelacion",
                  labels={"rate": "Tasa de cancelacion", col: "", "grupo": ""})
    fig3.update_xaxes(tickformat=".0%")
    fig3.update_yaxes(type="category")
    fig3.update_traces(textposition="outside")
    fig3.update_layout(height=max(340, 22 * len(det)))
    fig3.show()

    print(f"max_rate={max_rate:.3f} | hi_cut={hi_cut:.3f} lo_cut={lo_cut:.3f} | "
          f"KEEP alto={len(hi)}, KEEP bajo={len(lo)}  ->  {int(g.shape[0])} cats -> ~{n_keep+extra} columnas")
    return resumen

### 13.1 `agent` (agencia)

In [70]:
reducir_cardinalidad("agent")

max_rate=1.000 | hi_cut=0.600 lo_cut=0.300 | KEEP alto=17, KEEP bajo=38  ->  333 cats -> ~57 columnas


,_grupo,reservas,tasa_cancelacion,pct_reservas
0,Desconocido (nulo),16340,0.246634,0.136862
1,KEEP alto riesgo,13452,0.733497,0.112673
2,KEEP baja cancelacion,24885,0.160619,0.208435
3,Otros (agrupado),64713,0.406873,0.542030


De **333** agencias pasamos a **~57** columnas. Conservamos las de **alto riesgo** (varias al 80-100 %, las especulativas de §9.3) y las **muy fiables** (<30 % del máximo). El grupo `Otros` (~54 % de las reservas) cancela ≈ la tasa base: agencias sin señal extrema, agruparlas no pierde información útil. `Desconocido` (sin agencia, ~14 %) cancela bastante menos → es señal por sí mismo y lo mantenemos aparte.

### 13.2 `country` (país)

In [71]:
reducir_cardinalidad("country")

max_rate=0.566 | hi_cut=0.340 lo_cut=0.170 | KEEP alto=10, KEEP bajo=4  ->  177 cats -> ~16 columnas


,_grupo,reservas,tasa_cancelacion,pct_reservas
0,Desconocido (nulo),488,0.137295,0.004087
1,KEEP alto riesgo,57500,0.538487,0.481615
2,KEEP baja cancelacion,8032,0.164094,0.067275
3,Otros (agrupado),53370,0.222522,0.447022


De **177** países a **~16** columnas. Como el máximo es **Portugal (~57 %)**, el umbral adaptativo **sí conserva Portugal** y demás mercados de alta cancelación, además de los muy fiables (Alemania, Japón… <17 %). El grupo `Otros` cancela ≈ base: países de volumen medio sin sesgo claro.

### 13.3 `company` (empresa)

En `company` **imputamos los nulos a una categoría explícita `no_company`** en lugar de tratarlos como "desconocido": el hueco **no es un dato perdido**, significa *reserva sin empresa* (un estado real, equivalente a `has_company = False`; ver §5). Así `no_company` pasa a ser su propia columna one-hot con su tasa de cancelación.

In [72]:
reducir_cardinalidad("company", null_label="no_company")

max_rate=0.655 | hi_cut=0.393 lo_cut=0.197 | KEEP alto=1, KEEP bajo=9  ->  352 cats -> ~12 columnas


,_grupo,reservas,tasa_cancelacion,pct_reservas
0,KEEP alto riesgo,267,0.655431,0.002236
1,KEEP baja cancelacion,2755,0.110708,0.023076
2,Otros (agrupado),3775,0.188344,0.031619
3,no_company,112593,0.382200,0.943069


`company` está **94 % vacía**, así que `no_company` domina (y, de §5, esas reservas *sin empresa* cancelan **más**, ≈ tasa base). Aun así, **conservar empresas concretas aporta**: hay empresas que cancelan mucho (la 67 ≈ 66 %) y otras casi nada (<20 %). Por eso la propuesta para `company` es **doble**: `no_company` / `has_company` (§5) **y** estas pocas empresas con señal —premiando a las fiables y penalizando a las que cancelan—, en vez de descartar la columna entera.

**Resumen — reducción de cardinalidad.**

| Variable | Categorías | → columnas | Cobertura KEEP | En `Otros` |
|---|---|---|---|---|
| `agent`   | 333 | ~57 | ~32 % | ~54 % |
| `country` | 177 | ~16 | ~55 % | ~45 % |
| `company` | 352 | ~12 | ~3 %  | ~3 % (94 % nulos) |

La técnica recorta la dimensionalidad **conservando la señal** (categorías de riesgo extremo) y mete el "ruido de cola larga" en `Otros`. **Propuesta para `src/`** (no hecha aquí), a implementar como transformación **fit-on-train** para evitar fuga. Para `company`, además, implica **re-incorporarla** al modelo (hoy está descartada).

## 14. Conclusiones y decisiones

De este EDA salen las decisiones que guían el resto del proyecto (y, más tarde, el paquete `src/`). Mapeamos cada **hallazgo → decisión**:

**Partición y métrica**
- Clase desbalanceada (~37 % cancela, y muy distinta por hotel) → **partición estratificada** (`stratify`).
- Por el desbalanceo, el *accuracy* engaña → métrica principal **ROC-AUC**.

**Columnas a descartar**
- **Fuga de información**: fuera `reservation_status` y `reservation_status_date` (revelan el desenlace; no existen al predecir).
- **Fuga sutil**: `required_car_parking_spaces` separa de forma *perfecta* (0 cancelaciones de 7.416) porque la plaza se asigna en el **check-in** → información del futuro. Candidata a eliminar (§11.14). *(El pipeline `src/` aún la incluye; pendiente de quitar.)*
- **No generaliza / confundida con la estación**: fuera `arrival_date_year` (años parciales y desplazados). La estacionalidad la dan `arrival_date_month` y `arrival_date_week_number`.

**Filas a sanear**
- El año está limpio (no hay fechas imposibles), pero hay **~180 reservas sin huéspedes** y **~715 sin noches** (registros sin sentido) → **eliminarlas** (el pipeline ya quita las de 0 huéspedes).
- Un par de `adr` **extremas** (−6,4 y 5400) → *outliers* a vigilar o recortar.

**Filas duplicadas**
- ~34 % de las filas son **duplicados exactos**, pero son **reservas reales en bloque** (sin ID, etiqueta coherente, perfil de *Groups*/`Non Refund`), no errores → **se mantienen**.
- Riesgo documentado: como cancelan más (~58 %), una partición al azar mete copias en train y test → **métricas optimistas** (*fuga por duplicación*). Limitación conocida.

**Codificación**
- **One-hot** de las categóricas con cardinalidad acotada (`deposit_type`, `market_segment`, `customer_type`, `distribution_channel`, `country` recortado…), que es donde vive la señal fuerte.

**Candidatas a *feature engineering* (propuesta para `src/`, no hecho aquí)**
- `has_company` → **protege** contra la cancelación (el nulo de `company` es informativo; evita el *one-hot* de 352 IDs).
- `has_agent` → **riesgo**, pero **ojo: confundida con el hotel** → tratarla con cuidado.

**Observación de fondo**
- **City y Resort se comportan tan distinto** (volumen, antelación, estacionalidad, canal, tasa de cancelación) que casi podrían **modelarse por separado**. Como mínimo, conviene mirar todo *por hotel*.

> Recordatorio: esto es **EDA**. Las features `has_*` son una **propuesta**; el pipeline real vive en `src/` y se decide allí.